# Memory-Efficient Transformer Training Techniques

## Experimental Setup

To evaluate various optimization strategies, a controlled training environment
was established using the WikiText-2 (raw) dataset, a standard benchmark for
language modeling. The text data was tokenized with a strict maximum sequence
length of 1024 tokens.

The model utilized for these experiments was a custom, scaled-down configuration
of the Mistral architecture, featuring 6 hidden layers, a hidden size of 768,
and 12 attention heads. This reduced architecture allowed for rapid iteration
while maintaining the structural characteristics of larger LLMs.

All experiments were conducted over a single training epoch with a constant
learning rate of 5e-4.

Hardware Used:
- GPU: NVIDIA A100 Tensor Core GPU (via Google Colab)

## Results Tables

The following table summarizes the key performance metrics recorded across the
five experimental configurations.

|Experiment            |Batch Size|Peak Memory (GB)|Total Time (s)|Step Time (approx)|Perplexity|
|----------------------|----------|----------------|--------------|------------------|----------|
|Baseline (FP16)       |32        |34.35           |448.81        |~0.39 s           |125.86    |
|BF16 Precision        |32        |34.35           |448.87        |~0.39 s           |127.00    |
|Flash Attention 2     |56        |34.33           |255.91        |~0.39 s*          |136.66    |
|Sliding Window        |56        |34.33           |253.30        |~0.38 s           |135.08    |
|Gradient Checkpointing|80        |32.64           |536.61        |~1.17 s           |144.18    |

## Analysis & Discussion

The experiments on the NVIDIA A100 reveal distinct trade-offs between
throughput, memory efficiency, and convergence speed.

1. Precision Formats (Baseline vs. BF16): The switch from FP16 to BF16
   (bfloat16) showed negligible difference in memory usage (34.35 GB) and total
   training time. This is expected on an A100, which has dedicated Tensor Cores
   optimized for both formats. The slight increase in perplexity s might be a
   characteristic of BF16's reduced mantissa precision.

2. Flash Attention 2 demonstrated the most significant performance gain. By
   optimizing memory access patterns (IO-awareness) and utilizing the A100's
   high memory bandwidth, it reduced total training time by ~43% (448s $\to$
   255s). Crucially, it allowed for a 75% increase in batch size (32 $\to$ 56)
   without increasing peak memory. This confirms that for standard attention
   mechanisms, memory bandwidth and the overhead of storing large attention
   matrices are the primary bottlenecks, both of which Flash Attention
   mitigates.

3. Sliding Window Attention: This technique performed nearly identically to
   Flash Attention 2 (253s vs 255s). Given the sequence length of only 1024
   tokens, the computational cost of the full attention mechanism is already low
   when optimized by Flash Attention. Sliding Window attention typically shines
   at much longer sequences (e.g., 4k, 8k+ tokens) where the quadratic
   complexity of global attention becomes prohibitive.

4. Gradient Checkpointing achieved the lowest memory footprint (32.64 GB)
   despite running the largest batch size (80). However, this efficiency came at
   a high cost: training time increased by ~20% compared to the baseline and was
   more than double the time of the Flash Attention run. This technique forces
   the GPU to re-compute activations during the backward pass rather than
   storing them, effectively trading compute time for memory space. On a
   powerful card like the A100, this is often unnecessary unless training
   extremely large models (70B+) that physically cannot fit in VRAM otherwise.

5. A clear trend emerged where larger batch sizes led to worse (higher)
   perplexity (125.86 $\to$ 144.18). Since the experiment was fixed to 1 epoch,
   increasing the batch size resulted in fewer total training steps (updates).